In [0]:
from pyspark.sql.functions import col ,lit , current_timestamp, max , min , sum , count , trim , lower ,upper 

In [0]:
bronze_table = "globalretail.bronze.stores_raw"
silver_table = "globalretail.silver.dim_store"
hwm_table = "globalretail.validation.silver_watermarks"


hwm_last_loaded_at = spark.sql(f"SELECT last_successful_load_at from {hwm_table} where entity_name = 'stores'").collect()[0][0]

max_last_loaded_at = spark.sql(f" SELECT max(ingested_at) from {bronze_table} where ingested_at > '{hwm_last_loaded_at}'").collect()[0][0]

print(hwm_last_loaded_at)

print(max_last_loaded_at)


bronze_store_df = spark.read.table(bronze_table)

new_row_df = bronze_store_df.filter(col("ingested_at")> hwm_last_loaded_at).filter(col("ingested_at") <= max_last_loaded_at)

display(new_row_df)

In [0]:
new_row_df.createOrReplaceTempView("vw_stores_from_bronze")

In [0]:

spark.sql(f"""
          
          MERGE INTO globalretail.silver.dim_store as t
          USING (
              SELECT  abs(xxhash64(cast(store_id AS STRING))) AS store_sk,
              cast(store_id as BIGINT) as store_id,
              trim(store_name) as store_name,
              trim(city) as city,
              upper(trim(country_code)) as country_code,
              trim(timezone_name) as timezone_name,
              upper(trim(region)) as region,
              cast(opened_date as DATE) as opened_date,
              cast(updated_at as TIMESTAMP) as updated_at,
              is_active,
              cast(ingested_at as TIMESTAMP) as ingested_at
              FROM vw_stores_from_bronze
          ) as s

          on t.store_id = s.store_id
          WHEN MATCHED THEN UPDATE SET *
          WHEN NOT MATCHED THEN INSERT *
          
          """)

In [0]:

if max_last_loaded_at is not None:
    spark.sql(f"""
    UPDATE globalretail.validation.silver_watermarks
    SET   last_successful_load_at =  TIMESTAMP('{max_last_loaded_at}')
    WHERE entity_name = 'stores'
    """)
else:
    print("No new records to process — watermark not updated.")